# 06 — Robustness and sensitivity (R2–R8, §6.2, §6.3)

Runs the pre-specified robustness set **in full**, as frozen, despite the notebook 05
primary being non-significant and in the direction opposite to the prediction
(delta = +0.281, p = 0.382, min q = 0.959).

Running it anyway is the point of pre-specifying it: declining the robustness suite
because the primary disappointed would be selective reporting.

**One check is degenerate and is reported as such.** R2 was frozen as "median split
within ALT+" when the primary was the §4a tertile. The §4b guard fired, so the median
split *is* the primary. R2 therefore duplicates the primary and carries no independent
information; R3 (tertile within ALT+, 5v11) is the meaningful distributional sensitivity.

**R7 convention.** Arm assignments are held fixed and one line is dropped per refit, so
each delta measures that observation's influence. Recomputing the split on n=15 would
conflate observation influence with cutpoint movement.

## 1 — Header

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ -- tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
from src.utils import log_versions, run_timestamp, record_provenance, write_report, write_sha256_manifest

RUN_TIMESTAMP = run_timestamp()
VERSIONS = log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'matplotlib'])
print(f'Run: {RUN_TIMESTAMP}')
assert sys.version_info[:2] == (3, 12), f'Expected Python 3.12, got {sys.version_info[:2]}'
print(f'Python: {sys.version.split()[0]}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-bir-dependency


Library versions:
  pandas: 2.3.3
  numpy: 2.5.4
  scipy: 1.18.1
  sklearn: 1.9.1
  matplotlib: 3.11.2
Run: 2026-10-12T00:56:53Z
Python: 3.12.13


## 2 — Inputs from notebooks 04 and 05

In [2]:
from src.utils import PROCESSED_DIR
from src.cohort import build_cohort, assert_all, gold_only
from src.stratify import (
    LOW_LABEL, INTACT_LABEL, load_expression, stratify_median,
    stratify_tertile_within, TCAB1_GENE, osteosarcoma_rna_models,
)
from src.dependency_stats import (
    BIR_GENES, HR_GENES, SIX_GENES, load_gene_effects, load_all_gene_effects_for_line,
    composite_score, compare_groups, cliffs_delta, standardize_within_line,
    delta_difference_permutation,
)
from scipy import stats

strata = pd.read_csv(PROCESSED_DIR / 'tcab1_strata.csv', index_col='ModelID')
strata_neg = pd.read_csv(PROCESSED_DIR / 'tcab1_strata_alt_negative.csv', index_col='ModelID')

cohort = build_cohort(); assert_all(cohort)
alt_ids = cohort.alt_positive['ModelID'].tolist()
neg_ids = cohort.alt_negative['ModelID'].tolist()
names = pd.concat([cohort.alt_positive, cohort.alt_negative]).set_index('ModelID')['StrippedCellLineName']

effects = load_gene_effects([*alt_ids, *neg_ids], SIX_GENES)
composite = composite_score(effects)

PRIMARY_ARM = strata['arm_primary_8v8']
primary_res = compare_groups(
    composite.loc[PRIMARY_ARM.index[PRIMARY_ARM == LOW_LABEL]],
    composite.loc[PRIMARY_ARM.index[PRIMARY_ARM == INTACT_LABEL]],
    name='primary_median_8v8',
)
print(f"primary recomputed: delta = {primary_res.cliffs_delta:+.4f}  p = {primary_res.p_value:.4f}")

robustness = {}

def record(key, label, res, note=''):
    robustness[key] = {
        'check': key, 'description': label,
        'n_low': res.n_low, 'n_intact': res.n_intact,
        'p_value': res.p_value, 'cliffs_delta': res.cliffs_delta,
        'delta_ci_low': res.delta_ci_low, 'delta_ci_high': res.delta_ci_high,
        'hodges_lehmann': res.hodges_lehmann,
        'delta_magnitude': res.delta_magnitude,
        'direction_as_predicted': res.direction_as_predicted,
        'note': note,
    }
    print(f'{key:<6s} {label:<46s} n={res.n_low}v{res.n_intact:<3d} '
          f'delta={res.cliffs_delta:+.4f} p={res.p_value:.4f} pred={res.direction_as_predicted}')

primary recomputed: delta = +0.2812  p = 0.3823


## 3 — R1 gold-only, R2 median, R3 tertile-within

In [3]:
gold_ids = set(gold_only(cohort.alt_positive)['ModelID'])

# R1 -- gold-only within the primary arms
low = composite.loc[[m for m in PRIMARY_ARM.index if PRIMARY_ARM[m]==LOW_LABEL and m in gold_ids]]
intact = composite.loc[[m for m in PRIMARY_ARM.index if PRIMARY_ARM[m]==INTACT_LABEL and m in gold_ids]]
record('R1', 'gold-only ALT+ within primary arms', compare_groups(low, intact, name='R1'),
       'n=3 low; at 3v7 the exact null median |delta| is 0.33, so unstable')

# R2 -- degenerate: the guard made the median split the primary
record('R2', 'median split within ALT+ (DEGENERATE)', primary_res,
       'Identical to the primary: the §4b guard made the median split the primary rule. '
       'Carries no independent information.')

# R3 -- tertile computed within ALT+ only
wrap53 = load_expression(sorted(set(osteosarcoma_rna_models()) | set(alt_ids)), [TCAB1_GENE])[TCAB1_GENE]
arm_r3 = stratify_tertile_within(alt_ids, wrap53)
record('R3', 'tertile within ALT+ only (5v11)',
       compare_groups(composite.loc[arm_r3.index[arm_r3==LOW_LABEL]],
                      composite.loc[arm_r3.index[arm_r3==INTACT_LABEL]], name='R3'))

R1     gold-only ALT+ within primary arms             n=3v7   delta=-0.6190 p=0.1833 pred=True
R2     median split within ALT+ (DEGENERATE)          n=8v8   delta=+0.2812 p=0.3823 pred=False


R3     tertile within ALT+ only (5v11)                n=5v11  delta=+0.5636 p=0.0897 pred=False


## 4 — R4 continuous, R5 ATRX

In [4]:
# R4 -- continuous Spearman. Prediction: rho > 0 (higher WRAP53 -> less negative composite)
w_alt = wrap53.loc[alt_ids]
rho_r4 = stats.spearmanr(w_alt.to_numpy(), composite.loc[alt_ids].to_numpy())
r4 = {'check': 'R4', 'description': 'Spearman(WRAP53 expression, composite) across ALT+',
      'n': len(alt_ids), 'rho': float(rho_r4.statistic), 'p_value': float(rho_r4.pvalue),
      'prediction': 'rho > 0', 'as_predicted': bool(rho_r4.statistic > 0)}
print(f"R4     Spearman(WRAP53, composite) n={len(alt_ids)}  rho={r4['rho']:+.4f}  "
      f"p={r4['p_value']:.4f}  predicted rho>0: {r4['as_predicted']}")

# R5 -- ATRX-wildtype only (prespec §7 recorded this as expected near-vacuous)
atrx = cohort.alt_positive.set_index('ModelID')['atrx_status']
wt_ids = [m for m in alt_ids if atrx[m] != 'altered']
arm_r5 = PRIMARY_ARM.loc[wt_ids]
record('R5', f'ATRX-wildtype ALT+ only ({len(wt_ids)} of {len(alt_ids)})',
       compare_groups(composite.loc[arm_r5.index[arm_r5==LOW_LABEL]],
                      composite.loc[arm_r5.index[arm_r5==INTACT_LABEL]], name='R5'),
       f'Near-vacuous as prespec §7 anticipated: only {len(alt_ids)-len(wt_ids)} ALT+ line is '
       f'ATRX-altered, so this drops {len(alt_ids)-len(wt_ids)} line. Reported as '
       f'UNINFORMATIVE, not as evidence of no ATRX confounding.')

R4     Spearman(WRAP53, composite) n=16  rho=-0.4415  p=0.0869  predicted rho>0: False
R5     ATRX-wildtype ALT+ only (15 of 16)             n=7v8   delta=+0.1786 p=0.6126 pred=False


## 5 — R6 / §6.2 ALT− negative control and selectivity

In [5]:
ARM_NEG = strata_neg['arm_median']
neg_res = compare_groups(
    composite.loc[ARM_NEG.index[ARM_NEG == LOW_LABEL]],
    composite.loc[ARM_NEG.index[ARM_NEG == INTACT_LABEL]],
    name='alt_negative',
)
record('R6', 'ALT- negative control (lineage control per §7)', neg_res,
       'At 4v5 the exact null median |delta| is 0.30 and P(|delta|>=0.474)=0.286, so this '
       'is too small to gate on -- prespec §6.2.')

interaction = delta_difference_permutation(
    composite.loc[PRIMARY_ARM.index[PRIMARY_ARM == LOW_LABEL]],
    composite.loc[PRIMARY_ARM.index[PRIMARY_ARM == INTACT_LABEL]],
    composite.loc[ARM_NEG.index[ARM_NEG == LOW_LABEL]],
    composite.loc[ARM_NEG.index[ARM_NEG == INTACT_LABEL]],
)
print()
for k, v in interaction.items():
    print(f'  {k}: {v}')

R6     ALT- negative control (lineage control per §7) n=4v5   delta=-0.4000 p=0.4127 pred=True

  delta_alt_positive: 0.28125
  delta_alt_negative: -0.4
  delta_difference: 0.68125
  p_value: 0.1762
  n_perm: 5000
  selectivity_as_predicted: False
  alt_negative_p_value: 0.4126984126984127
  alt_negative_mwu_method: exact
  alt_negative_contradicts: False


## 6 — R7 leave-one-line-out

In [6]:
loo_rows = []
low_ids = PRIMARY_ARM.index[PRIMARY_ARM == LOW_LABEL].tolist()
intact_ids = PRIMARY_ARM.index[PRIMARY_ARM == INTACT_LABEL].tolist()
for dropped in alt_ids:
    l = [m for m in low_ids if m != dropped]
    i = [m for m in intact_ids if m != dropped]
    d = cliffs_delta(composite.loc[l], composite.loc[i])
    loo_rows.append({'dropped': dropped, 'cell_line': names[dropped],
                     'arm': PRIMARY_ARM[dropped], 'delta': d})
loo = pd.DataFrame(loo_rows).sort_values('delta')
print(loo.round(4).to_string(index=False))
deltas = loo['delta'].tolist()
r7 = {'check': 'R7', 'description': 'leave-one-line-out on the primary',
      'n_refits': len(deltas), 'delta_min': float(min(deltas)), 'delta_max': float(max(deltas)),
      'observed_delta': float(primary_res.cliffs_delta),
      'any_sign_flip_to_predicted': bool(any(d < 0 for d in deltas)),
      'all_same_sign_as_observed': bool(all((d > 0) == (primary_res.cliffs_delta > 0) for d in deltas))}
print(f"\nR7     delta range [{r7['delta_min']:+.4f}, {r7['delta_max']:+.4f}]  "
      f"observed {r7['observed_delta']:+.4f}")
print(f"       any refit reaching the PREDICTED direction: {r7['any_sign_flip_to_predicted']}")

   dropped       cell_line          arm  delta
ACH-003171          NRHOS1    TCAB1-low 0.1786
ACH-003185          NRHOS4    TCAB1-low 0.1786
ACH-003573            C337    TCAB1-low 0.1786
ACH-000082 G292CLONEA141B1 TCAB1-intact 0.2143
ACH-002433   CCLFPEDS0019T    TCAB1-low 0.2143
ACH-000410           SAOS2 TCAB1-intact 0.2500
ACH-002471           OS052 TCAB1-intact 0.2500
ACH-003178          NRHOS2    TCAB1-low 0.2500
ACH-001526            HUO9 TCAB1-intact 0.2857
ACH-002067            NOS1 TCAB1-intact 0.2857
ACH-003587          531MII TCAB1-intact 0.2857
ACH-002690           OS526 TCAB1-intact 0.3214
ACH-000364            U2OS TCAB1-intact 0.3571
ACH-002834           OS384    TCAB1-low 0.3571
ACH-001715           CAL72    TCAB1-low 0.4286
ACH-002688           OS457    TCAB1-low 0.4643

R7     delta range [+0.1786, +0.4643]  observed +0.2812
       any refit reaching the PREDICTED direction: False


## 7 — R8 drop SMARCAL1 from the BIR set

In [7]:
BIR_NO_SMARCAL1 = tuple(g for g in BIR_GENES if g != 'SMARCAL1')
comp_r8 = composite_score(effects, bir=BIR_NO_SMARCAL1, hr=HR_GENES)
record('R8', f'BIR set without SMARCAL1 ({", ".join(BIR_NO_SMARCAL1)})',
       compare_groups(comp_r8.loc[low_ids], comp_r8.loc[intact_ids], name='R8'),
       'Guards against the composite restating the published broad SMARCAL1 ALT dependency.')

R8     BIR set without SMARCAL1 (POLD3, RAD52)        n=8v8   delta=+0.1562 p=0.6454 pred=False


## 8 — §6.3 within-line standardised composite

In [8]:
t0 = time.time()
all_effects = load_all_gene_effects_for_line([*alt_ids, *neg_ids])
print(f'loaded genome-wide effects: {all_effects.shape}  ({time.time()-t0:.0f}s)')
z = standardize_within_line(all_effects)
z_cols = {g: next(c for c in z.columns if c.split(" (")[0].strip() == g) for g in SIX_GENES}
z_six = z[[z_cols[g] for g in SIX_GENES]].rename(columns={v: k for k, v in z_cols.items()})
comp_z = composite_score(z_six)
record('6.3', 'within-line z-scored composite',
       compare_groups(comp_z.loc[low_ids], comp_z.loc[intact_ids], name='6.3'),
       'Stronger control for line-level essentiality scale than the composite difference alone.')

loaded genome-wide effects: (25, 18531)  (3s)
6.3    within-line z-scored composite                 n=8v8   delta=+0.2812 p=0.3823 pred=False


## 9 — Persist and report

In [9]:
import json as _json

rob = pd.DataFrame(robustness).T.reset_index(drop=True)
rob_path = PROCESSED_DIR / 'robustness_table.csv'
rob.to_csv(rob_path, index=False)

loo_path = PROCESSED_DIR / 'loo_deltas.csv'
loo.to_csv(loo_path, index=False)

extras_path = PROCESSED_DIR / 'robustness_extras.json'
extras_path.write_text(_json.dumps({
    'R4_continuous': r4, 'R7_summary': r7,
    'interaction_6_2': interaction,
    'primary_recomputed': primary_res.as_dict(),
}, indent=2, sort_keys=True) + '\n')

lines = [
    '# Notebook 06 — Robustness and sensitivity',
    '',
    'The pre-specified robustness set, run **in full** as frozen, despite the notebook 05',
    'primary being non-significant and opposite in direction',
    f'(delta = {primary_res.cliffs_delta:+.4f}, p = {primary_res.p_value:.4f}).',
    '',
    'Sign convention: Cliff\'s delta **< 0** is the predicted direction (TCAB1-low more',
    'BIR-dependent relative to HR).',
    '',
    '## Summary table',
    '',
    '| Check | Description | n | delta | 95% CI | p | HL | Predicted direction? |',
    '|---|---|---|---|---|---|---|---|',
]
for key in ['R1', 'R2', 'R3', 'R5', 'R6', 'R8', '6.3']:
    r = robustness[key]
    lines.append(
        f"| **{key}** | {r['description']} | {r['n_low']}v{r['n_intact']} "
        f"| {r['cliffs_delta']:+.4f} | [{r['delta_ci_low']:+.3f}, {r['delta_ci_high']:+.3f}] "
        f"| {r['p_value']:.4f} | {r['hodges_lehmann']:+.4f} "
        f"| {'yes' if r['direction_as_predicted'] else 'no'} |"
    )
lines += [
    f"| **R4** | {r4['description']} | {r4['n']} | rho {r4['rho']:+.4f} | — "
    f"| {r4['p_value']:.4f} | — | {'yes' if r4['as_predicted'] else 'no'} |",
    f"| **R7** | leave-one-out range over {r7['n_refits']} refits "
    f"| — | [{r7['delta_min']:+.4f}, {r7['delta_max']:+.4f}] | — | — "
    f"| {'some refits reach it' if r7['any_sign_flip_to_predicted'] else 'no refit reaches it'} |",
    '',
    '## Notes attached to individual checks',
    '',
]
for key in ['R1', 'R2', 'R5', 'R6', 'R8', '6.3']:
    if robustness[key]['note']:
        lines += [f"**{key}.** {robustness[key]['note']}", '']
lines += [
    '## R7 leave-one-line-out detail',
    '',
    f"Arm assignments held fixed; one line dropped per refit. Observed primary delta "
    f"{r7['observed_delta']:+.4f}; refit range **[{r7['delta_min']:+.4f}, {r7['delta_max']:+.4f}]**.",
    '',
    '| Dropped | Cell line | Arm | delta without it |',
    '|---|---|---|---|',
]
for _, r in loo.iterrows():
    lines.append(f"| {r['dropped']} | {r['cell_line']} | {r['arm']} | {r['delta']:+.4f} |")
lines += [
    '',
    '## §6.2 ALT- selectivity and interaction',
    '',
    f"- Cliff's delta, ALT+ stratum: **{interaction['delta_alt_positive']:+.4f}**",
    f"- Cliff's delta, ALT- stratum: **{interaction['delta_alt_negative']:+.4f}**",
    f"- ALT- stratum's own two-sided p: **{interaction['alt_negative_p_value']:.4f}** "
    f"({interaction['alt_negative_mwu_method']})",
    f"- delta difference (ALT+ minus ALT-): **{interaction['delta_difference']:+.4f}**",
    f"- permutation p for the interaction ({interaction['n_perm']} shuffles within stratum): "
    f"**{interaction['p_value']:.4f}**",
    f"- §8.5 ALT- contradicts (significant same-direction shift): "
    f"**{interaction['alt_negative_contradicts']}**",
    '',
    'At 4v5 the exact null median |delta| is 0.30 with P(|delta| >= 0.474) = 0.286, so the',
    'ALT- stratum is too small to gate on. This is why §8.5 tests for a *significant*',
    'same-direction shift rather than comparing deltas (prespec §6.2, §8.5).',
    '',
    '## Files',
    '',
    '- `robustness_table.csv`',
    '- `loo_deltas.csv`',
    '- `robustness_extras.json`',
    '',
    'The §8/§9/§10 verdict is rendered in notebook 07, not here.',
]
report = write_report('06', lines)
manifest = write_sha256_manifest('06', [report, rob_path, loo_path, extras_path])
record_provenance([{
    'notebook': '06', 'run_timestamp': RUN_TIMESTAMP,
    'checks_run': sorted(robustness) + ['R4', 'R7'],
    'r7_delta_range': [r7['delta_min'], r7['delta_max']],
    'interaction_p': interaction['p_value'],
}])
print(f'wrote {report.name}, {manifest.name}')

wrote notebook_06_report.md, notebook_06_sha256.json
